In [1]:
import pandas as pd

df = pd.read_csv("calibration_results.csv")

df.head()

,dataset,model,ECE_raw,ECE_platt_scaling,ECE_isotonic_regression
0,coco,gdino,0.029604,0.004415,0.000184
1,cityscapes,gdino,0.062549,0.009556,0.000652
2,kitti,gdino,0.053031,0.000720,0.000176
3,pascal,gdino,0.026179,0.000699,0.000037
4,coco,yolov8l,0.098498,0.007959,0.001732


In [2]:
model_map = {
    "gdino": "Grounding DINO~\cite{liu2023grounding}",
    "yolov8l-worldv2": "YOLOv8-World v2~\cite{cheng2024yoloworld}",
    "owlvit": "OWL-ViT~\cite{minderer2022owlvit}",
    "rtdetr-l": "RT-DETR-L~\cite{lv2023detrs}",
    "yolov8l": "YOLOv8-L~\cite{yolov8_ultralytics}",
    "frcnn": "Faster R-CNN (ResNet 50)~\cite{ren2016fasterrcnn}"
    }

model_map_idx = {
    "gdino": 0,
    "yolov8l-worldv2": 1,
    "owlvit": 2,
    "rtdetr-l": 3,
    "yolov8l": 4,
    "frcnn": 5
    }

dataset_map = {
    "coco": "COCO",
    "cityscapes": "Cityscapes",
    "kitti": "KITTI",
    "pascal": "Pascal VOC"
}

df["model_idx"] = df["model"].map(model_map_idx)
df["model"] = df["model"].map(model_map)
df["dataset"] = df["dataset"].map(dataset_map)

In [5]:
def fmt(x):

    return f"{x:.1e}"  # scientific notation here, only one decimal place for better readability


def generate_latex_table(df):
    lines = []

    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(r"\caption{ECE before and after calibration using Platt Scaling (PS) and Isotonic Regression (IR).}")
    lines.append(r"\label{tab:calibration_results}")
    lines.append(r"\begin{tabular}{llccc}")
    lines.append(r"\toprule")
    lines.append(r"Dataset & Model & ECE (Raw) & ECE (PS) & ECE (IR) \\")
    lines.append(r"\midrule")

    # group by dataset
    first = True
    for dataset, group in df.groupby("dataset"):
        group = group.sort_values("model_idx")  # sort by model index for consistent order

        for i, (_, row) in enumerate(group.iterrows()):
            prefix = " & " if i == 0 else " & "

            if first: # with citations only for the first dataset, to avoid cluttering the table
                line = (
                    prefix +
                    f"{row['model']} & " 
                    fr"\num{{{fmt(row['ECE_raw'])}}} & "
                    fr"\num{{{fmt(row['ECE_platt_scaling'])}}} & "
                    fr"\num{{{fmt(row['ECE_isotonic_regression'])}}} \\"
                )
            else:
                line = (
                    prefix +
                    f"{row['model'].split('~')[0]} & " 
                    fr"\num{{{fmt(row['ECE_raw'])}}} & "
                    fr"\num{{{fmt(row['ECE_platt_scaling'])}}} & "
                    fr"\num{{{fmt(row['ECE_isotonic_regression'])}}} \\"
                )

            if i == 0:
                line = rf"\multirow{{{len(group)}}}{{*}}{{{dataset}}} " + line

            lines.append(line)

        first = False
        lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"  # replace last midrule

    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")

    return "\n".join(lines)

In [6]:
df = df[~df["model_idx"].isna()]  # filter to only include models in the mapping ( we also have results for soft vs. hard label training for another table)

latex_code = generate_latex_table(df)
print(latex_code)

\begin{table}[t]
\centering
\caption{ECE before and after calibration using Platt Scaling (PS) and Isotonic Regression (IR).}
\label{tab:calibration_results}
\begin{tabular}{llccc}
\toprule
Dataset & Model & ECE (Raw) & ECE (PS) & ECE (IR) \\
\midrule
\multirow{6}{*}{COCO}  & Grounding DINO~\cite{liu2023grounding} & \num{3.0e-02} & \num{4.4e-03} & \num{1.8e-04} \\
 & YOLOv8-World v2~\cite{cheng2024yoloworld} & \num{6.8e-02} & \num{2.1e-02} & \num{1.1e-03} \\
 & OWL-ViT~\cite{minderer2022owlvit} & \num{1.4e-02} & \num{2.5e-03} & \num{1.4e-04} \\
 & RT-DETR-L~\cite{lv2023detrs} & \num{1.0e-02} & \num{4.2e-03} & \num{1.7e-04} \\
 & YOLOv8-L~\cite{yolov8_ultralytics} & \num{9.8e-02} & \num{8.0e-03} & \num{1.7e-03} \\
 & Faster R-CNN (ResNet 50)~\cite{ren2016fasterrcnn} & \num{1.5e-02} & \num{3.9e-03} & \num{1.7e-04} \\
\midrule
\multirow{6}{*}{Cityscapes}  & Grounding DINO & \num{6.3e-02} & \num{9.6e-03} & \num{6.5e-04} \\
 & YOLOv8-World v2 & \num{5.5e-02} & \num{3.4e-02} & \num{2.9e-03} 